In [1]:
from __future__ import annotations

import sys

from pathlib import Path
from typing import List, Dict, Optional
import pandas as pd
import sys
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as cx
import folium
import mapclassify as mc



In [2]:
# GTFS_FOLDER = '/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/north_west_gtfs/'

In [3]:
# prep_gtfs_for_day_unzipped_hardcoded.py
# ------------------------------------------------------------
# Purpose:
#   Read UNZIPPED GTFS files from a folder, determine
#   services active on a chosen target date, filter trips +
#   stop_times to that date, and optionally to a time window
#   (e.g., 10:00–16:00).  Optionally restrict to buses
#   (route_type == 3).
#
# Hard-coded variables are defined below.
# ------------------------------------------------------------


# ============================================================
# -------- EDIT THESE VARIABLES BEFORE RUNNING ---------------
# ============================================================

GTFS_DIR   = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/north_west_gtfs")   # folder containing all GTFS .txt files
OUT_DIR    = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")         # where filtered CSVs will be saved
TARGET_DATE = "2025-11-07"                   # YYYY-MM-DD (representative weekday)
TIME_WINDOW = "10:00-16:00"                  # or None to keep full day
BUS_ONLY    = True                           # True = keep only bus routes
# ============================================================


In [4]:
################################# STEP 3 ####################################
#############################################################################

In [5]:
# ---------- helpers ----------

def hhmmss_to_minutes(tstr: str):
    if pd.isna(tstr):
        return None
    hh, mm, ss = map(int, str(tstr).split(":"))
    return hh * 60 + mm + (ss // 60)

def parse_time_window(tw: str):
    if not tw:
        return None, None
    start, end = tw.split("-")
    def to_min(x): h, m = map(int, x.split(":")); return 60*h + m
    return to_min(start), to_min(end)

def weekday_flag_for_date(date_str: str):
    dow = pd.Timestamp(date_str).dayofweek  # Monday=0
    return ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"][dow]

def read_gtfs_table(folder: Path, filename: str, dtype=None):
    p = folder / filename
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p, dtype=dtype)

def get_active_service_ids(calendar: pd.DataFrame,
                           cal_dates: pd.DataFrame,
                           target_date: str) -> pd.Series:
    """Return service_ids active on target_date."""
    base_ids = pd.Series([], dtype=object)
    if not calendar.empty:
        cal = calendar.copy()
        for col in ["start_date","end_date"]:
            if col in cal.columns:
                cal[col] = pd.to_datetime(cal[col], format="%Y%m%d", errors="coerce")
        cal["target"] = pd.to_datetime(target_date)
        wd = weekday_flag_for_date(target_date)
        if wd in cal.columns:
            mask = ((cal["target"] >= cal["start_date"]) &
                    (cal["target"] <= cal["end_date"]) &
                    (cal[wd] == 1))
            base_ids = cal.loc[mask, "service_id"].astype(str).drop_duplicates()

    added = pd.Series([], dtype=object)
    removed = pd.Series([], dtype=object)
    if not cal_dates.empty and {"date","exception_type","service_id"}.issubset(cal_dates.columns):
        cd = cal_dates.copy()
        cd["date"] = pd.to_datetime(cd["date"], format="%Y%m%d", errors="coerce")
        t = pd.to_datetime(target_date)
        todays = cd.loc[cd["date"] == t]
        added   = todays.loc[todays["exception_type"] == 1, "service_id"].astype(str).drop_duplicates()
        removed = todays.loc[todays["exception_type"] == 2, "service_id"].astype(str).drop_duplicates()

    active = pd.Index(base_ids).union(pd.Index(added))
    active = active.difference(pd.Index(removed))
    return pd.Series(active, dtype=object)


In [6]:




# ---------- main pipeline ----------

def prep_for_target_day(gtfs_dir: Path,
                        target_date: str,
                        time_window: str = None,
                        bus_only: bool = False,
                        out_dir: Path = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")):

    out_dir.mkdir(parents=True, exist_ok=True)

    routes      = read_gtfs_table(gtfs_dir, "routes.txt")
    trips       = read_gtfs_table(gtfs_dir, "trips.txt")
    stop_times  = read_gtfs_table(gtfs_dir, "stop_times.txt")
    calendar    = read_gtfs_table(gtfs_dir, "calendar.txt")
    cal_dates   = read_gtfs_table(gtfs_dir, "calendar_dates.txt")

    if trips.empty or stop_times.empty:
        print("Error: trips.txt or stop_times.txt missing or empty.", file=sys.stderr)
        sys.exit(1)

    # --- active services for chosen date
    active_service_ids = get_active_service_ids(calendar, cal_dates, target_date)
    if active_service_ids.empty:
        print(f"Warning: no active services found for {target_date}.", file=sys.stderr)
    active_service_ids.to_csv(out_dir / "active_service_ids.csv", index=False)

    # --- filter trips
    trips["service_id"] = trips["service_id"].astype(str)
    trips_active = trips[trips["service_id"].isin(set(active_service_ids))].copy()

    # fallback if only calendar_dates used
    if trips_active.empty and not cal_dates.empty:
        cd = cal_dates.copy()
        cd["date"] = pd.to_datetime(cd["date"], format="%Y%m%d", errors="coerce")
        cd = cd[(cd["date"] == pd.to_datetime(target_date)) & (cd["exception_type"] == 1)]
        if not cd.empty:
            sid = cd["service_id"].astype(str).unique()
            trips_active = trips[trips["service_id"].astype(str).isin(sid)].copy()

    if trips_active.empty:
        print(f"No trips active for {target_date}.", file=sys.stderr)
        sys.exit(2)

    # --- optional: bus-only
    if bus_only and not routes.empty and "route_type" in routes.columns:
        routes["route_type"] = pd.to_numeric(routes["route_type"], errors="coerce")
        bus_route_ids = routes.loc[routes["route_type"] == 3, "route_id"].astype(str).unique()
        trips_active = trips_active[trips_active["route_id"].astype(str).isin(bus_route_ids)]
        if trips_active.empty:
            print("Warning: bus_only removed all trips.", file=sys.stderr)

    trips_active.to_csv(out_dir / "trips_active.csv", index=False)

    # --- filter stop_times to active trips
    st = stop_times.merge(trips_active[["trip_id","route_id","service_id"]],
                          on="trip_id", how="inner")
    st["dep_minutes"] = st["departure_time"].apply(hhmmss_to_minutes)

    # --- time window
    start_min, end_min = parse_time_window(time_window)
    if start_min is not None and end_min is not None:
        st = st[(st["dep_minutes"] >= start_min) & (st["dep_minutes"] < end_min)]

    keep = ["trip_id","stop_id","stop_sequence","departure_time","dep_minutes","route_id","service_id"]
    st = st[keep].sort_values(["stop_id","dep_minutes","trip_id","stop_sequence"])
    st.to_csv(out_dir / "stop_times_active_window.csv", index=False)

    print(f"\nFiltered GTFS written to:\n  {out_dir/'active_service_ids.csv'}\n"
          f"  {out_dir/'trips_active.csv'}\n  {out_dir/'stop_times_active_window.csv'}\n")

# ------------------------------------------------------------
# run
# ------------------------------------------------------------
if __name__ == "__main__":
    prep_for_target_day(GTFS_DIR, TARGET_DATE, TIME_WINDOW, BUS_ONLY, OUT_DIR)



Filtered GTFS written to:
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/active_service_ids.csv
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/trips_active.csv
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/stop_times_active_window.csv



In [7]:
################################# STEP 4 ####################################
#############################################################################

In [8]:


# --- EDIT THESE ---
WORK_DIR = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")  # folder where stop_times_active_window.csv lives
WINDOW_HOURS = 6.0                 # e.g., 10:00–16:00 => 6 hours
STOPS_FILE = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/north_west_gtfs/stops.txt")  # optional: to add stop_name

# 1) Load filtered stop_times for your target date & window
st = pd.read_csv(WORK_DIR / "stop_times_active_window.csv")

# Safety: keep only the columns we need
needed = {"stop_id", "trip_id"}
missing = needed - set(st.columns)
if missing:
    raise ValueError(f"Missing columns in stop_times_active_window.csv: {missing}")

# 2) De-duplicate (stop_id, trip_id) so the same trip at the same stop is counted once
st_unique = st.drop_duplicates(subset=["stop_id", "trip_id"])

# 3) Count distinct trips per stop within the window
per_stop = (
    st_unique.groupby("stop_id")["trip_id"]
    .nunique()
    .rename("total_departures_in_window")
    .reset_index()
)

# 4) Convert to departures/hour
per_stop["departures_per_hour"] = per_stop["total_departures_in_window"] / WINDOW_HOURS

# 5) (Optional) add stop_name / parent_station for readability
try:
    stops = pd.read_csv(STOPS_FILE, dtype={"stop_id": str})
    # Ensure consistent dtype for merge keys
    per_stop["stop_id"] = per_stop["stop_id"].astype(str)
    stops["stop_id"] = stops["stop_id"].astype(str)

    cols = ["stop_id", "stop_name", "parent_station"]
    cols = [c for c in cols if c in stops.columns]
    per_stop = per_stop.merge(stops[cols], on="stop_id", how="left")
except FileNotFoundError:
    pass  # skips if you don't supply stops.txt

# 6) Save
out = WORK_DIR / "stop_departures_per_hour.csv"
per_stop.to_csv(out, index=False)
print(f"Written: {out}\nRows: {len(per_stop):,}")


Written: /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/stop_departures_per_hour.csv
Rows: 33,032


In [9]:
################################# STEP 5 ####################################
#############################################################################

In [10]:
# ================ EDIT THESE PATHS =================
GTFS_STOPS_FILE   = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/north_west_gtfs/stops.txt")                 # GTFS stops.txt
LSOA_BOUNDARY_FILE = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/Lower_layer_Super_Output_Areas/LSOA_2011_EW_BFC_V3.shp")        # LSOA 2011 polygons (BFC)
LSOA_LAD_LOOKUP    = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/LSOA2011_to_LSOA2021.csv")  # Lookup CSV (2011→2022)

OUT_DIR     = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")
OUT_CSV     = OUT_DIR / "stops_with_lsoa_lancashire.csv"
OUT_GEOJSON = OUT_DIR / "stops_with_lsoa_lancashire.geojson"

# British National Grid for spatial ops
PROJECTED_CRS = "EPSG:27700"
NEAREST_MAX_METERS = 300

# 14 LAD names for the Lancashire geography (12 districts + 2 unitaries)
LAD_NAMES_LANCASHIRE = {
    "Burnley","Chorley","Fylde","Hyndburn","Lancaster","Pendle","Preston",
    "Ribble Valley","Rossendale","South Ribble","West Lancashire","Wyre",
    "Blackpool","Blackburn with Darwen"
}
# ===================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
#Debug

In [12]:
# ---------- Load GTFS stops ----------
stops = pd.read_csv(GTFS_STOPS_FILE, dtype={"stop_id": str})
for col in ("stop_id","stop_lat","stop_lon"):
    if col not in stops.columns:
        raise ValueError(f"stops.txt is missing required column: {col}")

# basic sanity + GB bounds filter (catch swapped lat/lon)
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops = stops.dropna(subset=["stop_lon","stop_lat"])
gb_mask = stops["stop_lon"].between(-8.0, 2.5) & stops["stop_lat"].between(49.0, 61.5)
stops = stops.loc[gb_mask].copy()

# build GeoDataFrame in WGS84 → project to BNG
for c in ("geometry","index_left","index_right"):
    if c in stops.columns: stops.drop(columns=c, inplace=True, errors="ignore")

stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"]),
    crs="EPSG:4326"
).to_crs(PROJECTED_CRS)

# ---------- Load LSOA polygons ----------
lsoa = gpd.read_file(LSOA_BOUNDARY_FILE)
lsoa = lsoa.set_crs(PROJECTED_CRS) if lsoa.crs is None else lsoa.to_crs(PROJECTED_CRS)


# --- Pre-trim stops to a small buffer around the Lancashire LSOAs (speed + fewer unmatched) ---
study_area = lsoa.unary_union.buffer(2000)  # 2 km buffer around all LSOAs
stops_gdf = stops_gdf[stops_gdf.intersects(study_area)].copy()
print(f"Stops after pre-trim to Lancashire buffer: {len(stops_gdf):,}")





# find LSOA code/name columns (2011 layer typically LSOA11CD/LSOA11NM)
code_col = next((c for c in lsoa.columns if c.upper().startswith("LSOA11CD")), None)
name_col = next((c for c in lsoa.columns if c.upper().startswith("LSOA11NM")), None)
if code_col is None:
    raise ValueError("Could not find LSOA code column (expected like 'LSOA11CD') in boundary file.")
if name_col is None:
    name_col = code_col  # continue without name if absent

# ---------- Filter polygons to Lancashire via lookup ----------
if not LSOA_LAD_LOOKUP.exists():
    raise FileNotFoundError("Lookup CSV not found. Download the LSOA 2011→LAD 2022 lookup and set LSOA_LAD_LOOKUP path.")

look = pd.read_csv(LSOA_LAD_LOOKUP, dtype=str)
lsoa_code_lookup = next((c for c in look.columns if c.upper().startswith("LSOA11CD")), None)
lad_name_col     = next((c for c in look.columns if "LAD" in c.upper() and "NM" in c.upper()), None)
if not lsoa_code_lookup or not lad_name_col:
    raise ValueError("Lookup found, but couldn't detect 'LSOA11CD' and LAD name column (e.g., LAD22NM/LAD23NM).")

look = look[[lsoa_code_lookup, lad_name_col]].rename(columns={lsoa_code_lookup:"LSOA11CD", lad_name_col:"LADNM"})
look = look[look["LADNM"].isin(LAD_NAMES_LANCASHIRE)]
if look.empty:
    raise ValueError("Lookup filter produced zero rows. Check LAD_NAMES_LANCASHIRE vs lookup LAD names.")

# keep only Lancashire LSOAs
lsoa = lsoa.merge(look, left_on=code_col, right_on="LSOA11CD", how="inner")
if lsoa.empty:
    raise ValueError("LSOA layer empty after merge. Ensure boundary file is 2011 and matches lookup codes.")

# remove any conflicting sjoin column names
for c in ("index_left","index_right"):
    if c in lsoa.columns: lsoa.drop(columns=c, inplace=True, errors="ignore")

# ---------- Diagnostics ----------
print(f"CRS -> stops: {stops_gdf.crs}, lsoa: {lsoa.crs}")
print(f"Counts -> stops: {len(stops_gdf):,}, Lancashire LSOAs: {len(lsoa):,}")
print("Stops bounds (minx miny maxx maxy):", np.round(stops_gdf.total_bounds, 1))
print("LSOA  bounds (minx miny maxx maxy):", np.round(lsoa.total_bounds, 1))
print("LADs kept:", ", ".join(sorted(lsoa['LADNM'].unique())))

# ---------- Spatial join: within ----------
joined = gpd.sjoin(
    stops_gdf,
    lsoa[[code_col, name_col, "LADNM", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={code_col:"LSOA11CD", name_col:"LSOA11NM"})

matched_within = joined["LSOA11CD"].notna().sum()
print(f"Stops matched by 'within': {matched_within:,}")

# mark method
joined["join_method"] = np.where(joined["LSOA11CD"].notna(), "within", None)


# ---------- Nearest fallback (≤ NEAREST_MAX_METERS) ----------
unmatched = joined["LSOA11CD"].isna().sum()
if unmatched > 0:
    # rows still missing an LSOA
    missing = joined.loc[joined["LSOA11CD"].isna()].copy()

    # 1) Drop any columns that could trigger suffixing/clashes
    for c in ["LSOA11CD", "LSOA11NM", "LADNM", "nearest_dist_m", "join_method", "index_left", "index_right"]:
        if c in missing.columns:
            missing.drop(columns=c, inplace=True)

    # also make sure RHS doesn't carry sjoin artifacts
    lsoa_near = lsoa[[code_col, name_col, "LADNM", "geometry"]].copy()
    for c in ["index_left", "index_right"]:
        if c in lsoa_near.columns:
            lsoa_near.drop(columns=c, inplace=True)

    # 2) Run nearest join (no suffixes needed now)
    nearest = gpd.sjoin_nearest(
        missing,
        lsoa_near,
        how="left",
        max_distance=NEAREST_MAX_METERS,
        distance_col="nearest_dist_m",
    )

    # 3) Standardise column names for RHS attributes
    #    (if your boundary already uses LSOA11CD/LSOA11NM, code_col/name_col will equal those)
    rename_map = {}
    if code_col in nearest.columns and code_col != "LSOA11CD":
        rename_map[code_col] = "LSOA11CD"
    if name_col in nearest.columns and name_col != "LSOA11NM":
        rename_map[name_col] = "LSOA11NM"
    if rename_map:
        nearest.rename(columns=rename_map, inplace=True)

    # Safety: if suffixes somehow still appeared, handle them too
    for c_src, c_dst in [(f"{code_col}_right", "LSOA11CD"),
                         (f"{name_col}_right", "LSOA11NM"),
                         (f"{code_col}_lsoa", "LSOA11CD"),
                         (f"{name_col}_lsoa", "LSOA11NM")]:
        if c_src in nearest.columns and c_dst not in nearest.columns:
            nearest.rename(columns={c_src: c_dst}, inplace=True)

    got = int(nearest["LSOA11CD"].notna().sum())
    print(f"Stops matched by 'nearest' (≤{NEAREST_MAX_METERS} m): {got:,}")

    # 4) Write nearest matches back into the original 'joined' by left index
    match_mask = nearest["LSOA11CD"].notna()
    left_idx = nearest.index[match_mask]
    cols_to_copy = ["LSOA11CD", "LSOA11NM", "LADNM", "nearest_dist_m"]
    for col in cols_to_copy:
        if col not in nearest.columns:
            nearest[col] = pd.NA

    joined.loc[left_idx, ["LSOA11CD", "LSOA11NM", "LADNM"]] = \
        nearest.loc[match_mask, ["LSOA11CD", "LSOA11NM", "LADNM"]].values
    joined.loc[left_idx, "join_method"] = "nearest"
    joined.loc[left_idx, "nearest_dist_m"] = nearest.loc[match_mask, "nearest_dist_m"].values

still = int(joined["LSOA11CD"].isna().sum())
if still:
    print(f"Still unmatched after nearest: {still} (likely outside Lancashire boundary or > {NEAREST_MAX_METERS} m).")




# ---------- Save ----------
cols_out = ["stop_id","stop_name","stop_lat","stop_lon","LSOA11CD","LSOA11NM","LADNM","join_method","nearest_dist_m","geometry"]
cols_out = [c for c in cols_out if c in joined.columns]
gout = joined[cols_out].copy()

gout.drop(columns="geometry").to_csv(OUT_CSV, index=False)
gout.to_file(OUT_GEOJSON, driver="GeoJSON")

print(f"\nSaved:\n  {OUT_CSV}\n  {OUT_GEOJSON}")


/var/folders/d8/hkx8jz3s4_zbp6_5ff82k80h0000gn/T/ipykernel_78543/2469103733.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  study_area = lsoa.unary_union.buffer(2000)  # 2 km buffer around all LSOAs


Stops after pre-trim to Lancashire buffer: 35,817
CRS -> stops: EPSG:27700, lsoa: EPSG:27700
Counts -> stops: 35,817, Lancashire LSOAs: 950
Stops bounds (minx miny maxx maxy): [296772.  173498.5 534373.7 583047.2]
LSOA  bounds (minx miny maxx maxy): [330207.  398767.  397131.6 482748.7]
LADs kept: Blackburn with Darwen, Blackpool, Burnley, Chorley, Fylde, Hyndburn, Lancaster, Pendle, Preston, Ribble Valley, Rossendale, South Ribble, West Lancashire, Wyre
Stops matched by 'within': 7,485
Stops matched by 'nearest' (≤300 m): 157
Still unmatched after nearest: 28288 (likely outside Lancashire boundary or > 300 m).

Saved:
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/stops_with_lsoa_lancashire.csv
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/stops_with_lsoa_lancashire.geojson


In [13]:
################################# STEP 6 ####################################
#############################################################################

In [14]:
# step6_aggregate_to_lsoa.py
# ------------------------------------------------------------
# Inputs:
#   1) stop_departures_per_hour.csv   (from Step 4)
#        - stop_id, total_departures_in_window, departures_per_hour, [stop_name,...]
#   2) stops_with_lsoa_lancashire.csv (from Step 5)
#        - stop_id, LSOA11CD, LSOA11NM, LADNM, join_method, nearest_dist_m
#   3) LSOA_2011_EW_BFC.shp (or .geojson) for mapping output
#
# Outputs (in OUT_DIR):
#   - lsoa_bus_frequency.csv
#   - lsoa_bus_frequency.geojson
#
# What it computes per LSOA:
#   - stops_count
#   - avg_departures_per_hour (mean)
#   - median_departures_per_hour (primary Bus Frequency Score)
#   - pct_stops_lt1_per_hour (% stops with < 1 bus/hour)
#   - BusFreqBand (Desert / Low / Moderate / Good / High)
#   - BusFreq_norm (0–1, higher = worse; based on median)
# ------------------------------------------------------------

In [15]:
# =============== EDIT THESE PATHS =================
WORK_DIR = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")
PER_STOP_FREQ_FILE = WORK_DIR / "stop_departures_per_hour.csv"
STOPS_LSOA_FILE    = WORK_DIR / "stops_with_lsoa_lancashire.csv"
LSOA_BOUNDARY_FILE = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/Lower_layer_Super_Output_Areas/LSOA_2011_EW_BFC_V3.shp")  # or .geojson
OUT_DIR            = WORK_DIR
OUT_CSV            = OUT_DIR / "lsoa_bus_frequency.csv"
OUT_GEOJSON        = OUT_DIR / "lsoa_bus_frequency.geojson"
# =================================================


In [16]:



# ---------- 1) Load inputs ----------
per_stop = pd.read_csv(PER_STOP_FREQ_FILE, dtype={"stop_id": str})
stops_lsoa = pd.read_csv(STOPS_LSOA_FILE, dtype={"stop_id": str})

# Basic column checks
for col in ("stop_id", "departures_per_hour"):
    if col not in per_stop.columns:
        raise ValueError(f"{PER_STOP_FREQ_FILE.name} missing required column: {col}")
for col in ("stop_id", "LSOA11CD"):
    if col not in stops_lsoa.columns:
        raise ValueError(f"{STOPS_LSOA_FILE.name} missing required column: {col}")

# Keep only necessary columns and deduplicate stop_id just in case
per_stop = per_stop[["stop_id", "departures_per_hour"]].drop_duplicates(subset=["stop_id"])
stops_lsoa_small = stops_lsoa[["stop_id", "LSOA11CD", "LSOA11NM", "LADNM"]].drop_duplicates(subset=["stop_id"])

# ---------- 2) Merge per-stop frequency with LSOA code ----------
df = per_stop.merge(stops_lsoa_small, on="stop_id", how="inner")

# If you want to exclude stops that were assigned by 'nearest' in step 5, uncomment:
# df = df.merge(stops_lsoa[["stop_id","join_method"]], on="stop_id", how="left")
# df = df[df["join_method"].fillna("within").isin(["within"])].copy()

if df.empty:
    raise ValueError("After merging, no stops have both frequency and LSOA. Check inputs.")

# ---------- 3) Aggregate to LSOA ----------
def pct_lt1(series):
    return float((series < 1.0).mean() * 100.0)

agg = (
    df.groupby(["LSOA11CD", "LSOA11NM", "LADNM"], dropna=False)
      .agg(
          stops_count=("stop_id", "nunique"),
          avg_departures_per_hour=("departures_per_hour", "mean"),
          median_departures_per_hour=("departures_per_hour", "median"),
          pct_stops_lt1_per_hour=("departures_per_hour", pct_lt1),
      )
      .reset_index()
)

# ---------- 4) Classification bands (based on MEDIAN) ----------
def classify_band(median_val: float) -> str:
    if pd.isna(median_val):
        return "Unknown"
    if median_val < 1.0: return "Desert"
    if median_val < 2.0: return "Low"
    if median_val < 4.0: return "Moderate"
    if median_val < 6.0: return "Good"
    return "High"

agg["BusFreqBand"] = agg["median_departures_per_hour"].apply(classify_band)

# ---------- 5) Normalised score (0–1, higher = worse) ----------
# Use min/max of medians across LSOAs (ignore zeros-only areas by keeping all medians)
med = agg["median_departures_per_hour"]
mn, mx = float(med.min()), float(med.max())
if np.isfinite(mn) and np.isfinite(mx) and mx > mn:
    # invert so fewer buses => higher risk
    agg["BusFreq_norm"] = (mx - agg["median_departures_per_hour"]) / (mx - mn)
else:
    # fallback if all equal
    agg["BusFreq_norm"] = 0.0

# Order columns neatly
cols = [
    "LSOA11CD", "LSOA11NM", "LADNM",
    "stops_count",
    "avg_departures_per_hour",
    "median_departures_per_hour",
    "pct_stops_lt1_per_hour",
    "BusFreqBand",
    "BusFreq_norm",
]
agg = agg[cols].sort_values(["LADNM", "median_departures_per_hour"])

# ---------- 6) Save CSV ----------
OUT_DIR.mkdir(parents=True, exist_ok=True)
agg.to_csv(OUT_CSV, index=False)
print(f"Saved CSV: {OUT_CSV}  (rows: {len(agg):,})")

# ---------- 7) (Optional) Save GeoJSON for mapping ----------
# Join to LSOA polygons (2011) for choropleth
try:
    lsoa = gpd.read_file(LSOA_BOUNDARY_FILE)
    # find code col (usually LSOA11CD)
    code_col = next((c for c in lsoa.columns if c.upper().startswith("LSOA11CD")), None)
    if code_col is None:
        raise ValueError("Could not find LSOA code column in boundary file (expected like 'LSOA11CD').")

    # Merge attributes to geometry
    g = lsoa.merge(agg, left_on=code_col, right_on="LSOA11CD", how="inner")

    # Ensure a CRS exists (most ONS layers are already in EPSG:27700)
    if g.crs is None:
        g = g.set_crs("EPSG:27700")

    g.to_file(OUT_GEOJSON, driver="GeoJSON")
    print(f"Saved GeoJSON: {OUT_GEOJSON}  (features: {len(g):,})")
except Exception as e:
    print(f"(Skipping GeoJSON export) Reason: {e}")


Saved CSV: /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency.csv  (rows: 897)
Saved GeoJSON: /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency.geojson  (features: 896)


In [17]:
################################# STEP 7 ####################################
#############################################################################

In [18]:
# ---------- PATHS ----------

WORK_DIR = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")
LSOA_FILE = WORK_DIR / "lsoa_bus_frequency.csv"     # from Step 6
LSOA_BOUNDARY_FILE = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/Lower_layer_Super_Output_Areas/LSOA_2011_EW_BFC_V3.shp")  # for mapping
OUT_CSV = WORK_DIR / "lsoa_bus_frequency_scored.csv"
OUT_GEOJSON = WORK_DIR / "lsoa_bus_frequency_scored.geojson"



In [19]:
# step7_create_bus_frequency_score.py
# ------------------------------------------------------------
# Input:  lsoa_bus_frequency.csv  (from Step 6)
# Output: lsoa_bus_frequency_scored.csv  +  GeoJSON (optional)
# ------------------------------------------------------------



# ---------- 1. LOAD ----------
df = pd.read_csv(LSOA_FILE)

if "median_departures_per_hour" not in df.columns:
    raise ValueError("Missing column 'median_departures_per_hour' in input file.")

# ---------- 2. CREATE Bus Frequency SCORE ----------
df["BusFreqScore"] = df["median_departures_per_hour"]

# ---------- 3. CREATE NORMALISED (0–1) VERSION ----------
mn, mx = df["BusFreqScore"].min(), df["BusFreqScore"].max()
if np.isfinite(mn) and np.isfinite(mx) and mx > mn:
    df["BusFreq_norm"] = (mx - df["BusFreqScore"]) / (mx - mn)
else:
    df["BusFreq_norm"] = 0.0

# Interpretation: 0 = best (many buses), 1 = worst (fewest buses)

# ---------- 4. OPTIONAL: CLASSIFY (same bands for readability) ----------
def classify_band(score: float) -> str:
    if pd.isna(score): return "Unknown"
    if score < 1.0: return "Desert"
    if score < 2.0: return "Low"
    if score < 4.0: return "Moderate"
    if score < 6.0: return "Good"
    return "High"

if "BusFreqBand" not in df.columns:
    df["BusFreqBand"] = df["BusFreqScore"].apply(classify_band)

# ---------- 5. SAVE ----------
df.to_csv(OUT_CSV, index=False)
print(f"Saved CSV: {OUT_CSV}  (rows: {len(df):,})")

# ---------- 6. OPTIONAL: JOIN TO GEOMETRY FOR MAPPING ----------
try:
    gdf = gpd.read_file(LSOA_BOUNDARY_FILE)
    code_col = next((c for c in gdf.columns if c.upper().startswith("LSOA11CD")), None)
    if code_col:
        gdf = gdf.merge(df, left_on=code_col, right_on="LSOA11CD", how="inner")
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:27700")
        gdf.to_file(OUT_GEOJSON, driver="GeoJSON")
        print(f"Saved GeoJSON: {OUT_GEOJSON}  (features: {len(gdf):,})")
    else:
        print("No LSOA code column detected in boundary shapefile – skipping GeoJSON join.")
except Exception as e:
    print(f"(Skipping GeoJSON export) Reason: {e}")


Saved CSV: /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored.csv  (rows: 897)
Saved GeoJSON: /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored.geojson  (features: 896)


In [20]:
################################# STEP 8 ####################################
#############################################################################

In [21]:
# step8_quality_checks.py
# ------------------------------------------------------------
# Inputs:
#   - data/working/lsoa_bus_frequency_scored.csv   (from Step 7)
#   - data/working/stops_with_lsoa_lancashire.csv  (from Step 5)
#   - data/working/stop_times_active_window.csv    (from Step 3/4)  [optional for clockface]
#   - data/boundaries/LSOA_2011_EW_BFC.shp         (LSOA polygons)
#
# Outputs (in WORK_DIR):
#   - qc_ranked_lsoas.csv              (top/bottom lists for “sanity by eye”)
#   - qc_lad_level_summary.csv         (LAD-level medians)
#   - qc_clockface_stop_<id>.csv       (per stop headways if STOP_CHECK_IDS provided)
#   - qc_extreme_deserts.csv           (LSOAs with 0 stops -> Extreme Desert)
#   - lsoa_bus_frequency_scored_qc.csv (final table with extreme deserts filled)
# ------------------------------------------------------------

# ---------- EDIT THESE ----------
WORK_DIR = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")
BOUNDARY = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/Lower_layer_Super_Output_Areas/LSOA_2011_EW_BFC_V3.shp")  # or .geojson

LSOA_SCORED_FILE   = WORK_DIR / "lsoa_bus_frequency_scored.csv"
STOPS_LSOA_FILE    = WORK_DIR / "stops_with_lsoa_lancashire.csv"
STOP_TIMES_WINDOW  = WORK_DIR / "stop_times_active_window.csv"  # optional (for clockface)

# For the clockface check: add a few stop_ids you know well, or leave empty to skip.
STOP_CHECK_IDS = []  # e.g., ["1800IM11812","1800IM11813"]

# How many LSOAs to show in top/bottom lists
TOP_N = 20
# ---------------------------------

def ensure_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

# ---------- 1) Load inputs ----------
lsoa_scored = pd.read_csv(LSOA_SCORED_FILE, dtype={"LSOA11CD": str})
stops_lsoa  = pd.read_csv(STOPS_LSOA_FILE, dtype={"stop_id": str, "LSOA11CD": str})

ensure_cols(lsoa_scored,
            ["LSOA11CD","LSOA11NM","LADNM","stops_count","BusFreqScore","median_departures_per_hour"],
            LSOA_SCORED_FILE.name)
ensure_cols(stops_lsoa, ["stop_id","LSOA11CD"], STOPS_LSOA_FILE.name)

# ---------- 2) Sanity by eye ----------
# Rank LSOAs by BusFreqScore (median departures/hour)
ranked = lsoa_scored.sort_values("BusFreqScore", ascending=False).copy()
ranked["rank_desc"] = np.arange(1, len(ranked)+1)

# Export top/bottom lists
top_df = ranked.head(TOP_N).assign(list_type="top")
bot_df = ranked.tail(TOP_N).assign(list_type="bottom")
rank_out = pd.concat([top_df, bot_df], ignore_index=True)
rank_out.to_csv(WORK_DIR / "qc_ranked_lsoas.csv", index=False)

# LAD-level medians (should show city centres > rural)
lad_summary = (
    lsoa_scored
    .groupby("LADNM", as_index=False)["BusFreqScore"]
    .median()
    .rename(columns={"BusFreqScore":"LAD_median_departures_per_hour"})
    .sort_values("LAD_median_departures_per_hour", ascending=False)
)
lad_summary.to_csv(WORK_DIR / "qc_lad_level_summary.csv", index=False)

print(f"[Sanity] Saved ranked LSOAs to {WORK_DIR/'qc_ranked_lsoas.csv'}")
print(f"[Sanity] Saved LAD-level medians to {WORK_DIR/'qc_lad_level_summary.csv'}")

# ---------- 3) Clockface check (optional) ----------
if STOP_CHECK_IDS and STOP_TIMES_WINDOW.exists():
    st = pd.read_csv(STOP_TIMES_WINDOW, dtype={"stop_id": str})
    ensure_cols(st, ["stop_id","trip_id","departure_time","dep_minutes"], STOP_TIMES_WINDOW.name)

    for sid in STOP_CHECK_IDS:
        one = st.loc[st["stop_id"] == sid].copy()
        one = one.sort_values("dep_minutes")
        # compute headways (minutes between consecutive departures)
        one["headway_min"] = one["dep_minutes"].diff()
        out = WORK_DIR / f"qc_clockface_stop_{sid}.csv"
        one[["stop_id","trip_id","departure_time","dep_minutes","headway_min"]].to_csv(out, index=False)
        print(f"[Clockface] Wrote {out} (rows: {len(one)})")
else:
    print("[Clockface] Skipped (no STOP_CHECK_IDS or stop_times_active_window.csv not found).")

# ---------- 4) Stop coverage / Extreme deserts ----------
# Build the universe of Lancashire LSOAs from boundary file (align codes)
lsoa_poly = gpd.read_file(BOUNDARY)
code_col = next((c for c in lsoa_poly.columns if c.upper().startswith("LSOA11CD")), None)
name_col = next((c for c in lsoa_poly.columns if c.upper().startswith("LSOA11NM")), None)
if code_col is None:
    raise ValueError("Boundary file lacks an LSOA code column (expected like 'LSOA11CD').")
if name_col is None:
    name_col = code_col

# Subset boundary to just the LSOAs present in your scored table’s LADs (safer if you only kept Lancashire earlier)
lad_names = set(lsoa_scored["LADNM"].dropna().unique())
# If your boundary doesn’t carry LAD names, we’ll assume Step 5 already filtered to Lancashire.
if "LADNM" in lsoa_poly.columns:
    lsoa_poly = lsoa_poly[lsoa_poly["LADNM"].isin(lad_names)]  # keep only those LADs

universe = lsoa_poly[[code_col, name_col]].rename(columns={code_col:"LSOA11CD", name_col:"LSOA11NM"})
universe["LSOA11CD"] = universe["LSOA11CD"].astype(str)

# Left join universe to scored: missing rows are deserts with zero stops
scored_all = universe.merge(lsoa_scored, on=["LSOA11CD","LSOA11NM"], how="left")

# Fill extreme deserts (no stops_count -> 0; BusFreqScore -> 0; BusFreq_norm -> 1)
scored_all["stops_count"] = scored_all["stops_count"].fillna(0).astype(int)
scored_all["BusFreqScore"] = scored_all["BusFreqScore"].fillna(0.0)
if "BusFreq_norm" not in scored_all.columns:
    # If it wasn't there for some reason, create a conservative default: 1.0 for no-service areas, else rescale existing
    med = scored_all["BusFreqScore"]
    mn, mx = float(med.min()), float(med.max())
    if np.isfinite(mn) and np.isfinite(mx) and mx > mn:
        scored_all["BusFreq_norm"] = (mx - scored_all["BusFreqScore"]) / (mx - mn)
    else:
        scored_all["BusFreq_norm"] = 0.0
# explicitly set deserts (no stops) to worst score
scored_all.loc[scored_all["stops_count"] == 0, "BusFreq_norm"] = 1.0

# Add/override band with Extreme Desert where stops_count==0
def band_with_extreme(row):
    if row.get("stops_count", 0) == 0:
        return "Extreme Desert"
    # keep existing band if present, else classify off BusFreqScore
    band = row.get("BusFreqBand", None)
    if pd.isna(band) or band is None:
        v = row.get("BusFreqScore", np.nan)
        if pd.isna(v): return "Unknown"
        if v < 1.0: return "Desert"
        if v < 2.0: return "Low"
        if v < 4.0: return "Moderate"
        if v < 6.0: return "Good"
        return "High"
    return band

scored_all["BusFreqBand"] = scored_all.apply(band_with_extreme, axis=1)

extreme = scored_all.loc[scored_all["stops_count"] == 0].copy()
extreme.to_csv(WORK_DIR / "qc_extreme_deserts.csv", index=False)

# Save final QC’d table
scored_all.to_csv(WORK_DIR / "lsoa_bus_frequency_scored_qc.csv", index=False)

print(f"[Coverage] Extreme deserts written to {WORK_DIR/'qc_extreme_deserts.csv'} (rows: {len(extreme)})")
print(f"[Final] QC’d table written to {WORK_DIR/'lsoa_bus_frequency_scored_qc.csv'} (rows: {len(scored_all)})")

# ---------- 5) Quick textual summaries ----------
print("\n[Summary] Top 5 LSOAs by BusFreqScore:")
print(ranked[["LSOA11CD","LSOA11NM","LADNM","BusFreqScore"]].head(5).to_string(index=False))

print("\n[Summary] Bottom 5 LSOAs by BusFreqScore:")
print(ranked[["LSOA11CD","LSOA11NM","LADNM","BusFreqScore"]].tail(5).to_string(index=False))

print("\n[Summary] LAD medians (top 5):")
print(lad_summary.head(5).to_string(index=False))


[Sanity] Saved ranked LSOAs to /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/qc_ranked_lsoas.csv
[Sanity] Saved LAD-level medians to /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/qc_lad_level_summary.csv
[Clockface] Skipped (no STOP_CHECK_IDS or stop_times_active_window.csv not found).
[Coverage] Extreme deserts written to /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/qc_extreme_deserts.csv (rows: 33857)
[Final] QC’d table written to /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored_qc.csv (rows: 34753)

[Summary] Top 5 LSOAs by BusFreqScore:
 LSOA11CD       LSOA11NM     LADNM  BusFreqScore
E01012679 Blackpool 008B Blackpool     29.333333
E01033073 Lancaster 014F Lancaster     21.166667
E01024991     Fylde 00

In [22]:
################################# STEP 9 ####################################
#############################################################################

In [3]:
# ----------------- EDIT THESE -----------------
WORK_DIR = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output")
BOUNDARY = Path("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/LSOA_Boundaries/Lower_layer_Super_Output_Areas/LSOA_2011_EW_BFC_V3.shp")  # or .geojson

LSOA_SCORED_QC = WORK_DIR / "lsoa_bus_frequency_scored_qc.csv"  # from Step 8
STOPS_LSOA     = WORK_DIR / "stops_with_lsoa_lancashire.csv"    # from Step 5
PER_STOP_FREQ  = WORK_DIR / "stop_departures_per_hour.csv"      # from Step 4

OUT_PNG_CONT   = WORK_DIR / "map_busfreq_continuous.png"
OUT_PNG_CAT    = WORK_DIR / "map_busfreq_band.png"
OUT_HTML       = WORK_DIR / "map_busfreq_interactive.html"
OUT_PBI_CSV    = WORK_DIR / "map_export_powerbi.csv"
# -------------------------------------------------------------

In [ ]:
# step9_map_and_export.py
# ------------------------------------------------------------
# Creates static + interactive maps for Lancashire bus frequency.
# - Choropleth by BusFreqScore (continuous) and by BusFreqBand (categorical)
# - Overlay stops with < 1 departure/hour
# - Exports PNGs, HTML map, and a clean CSV for Power BI
# ------------------------------------------------------------




# 1) Load data
scores = pd.read_csv(LSOA_SCORED_QC, dtype={"LSOA11CD": str})
lsoa = gpd.read_file(BOUNDARY)
code_col = next((c for c in lsoa.columns if c.upper().startswith("LSOA11CD")), None)
name_col = next((c for c in lsoa.columns if c.upper().startswith("LSOA11NM")), None)
if code_col is None: raise ValueError("Boundary missing LSOA11CD column.")

# keep only LSOAs present in scores (Lancashire subset)
g = lsoa.merge(scores, left_on=code_col, right_on="LSOA11CD", how="inner")

# 2) Build stops layer for overlay (<1 dep/hr)
per_stop = pd.read_csv(PER_STOP_FREQ, dtype={"stop_id": str})
stops_lsoa = pd.read_csv(STOPS_LSOA, dtype={"stop_id": str})
stops = per_stop.merge(stops_lsoa[["stop_id","stop_lat","stop_lon"]], on="stop_id", how="left").dropna(subset=["stop_lat","stop_lon"])
stops_low = stops.loc[stops["departures_per_hour"] < 1.0].copy()
stops_low_gdf = gpd.GeoDataFrame(stops_low,
                                 geometry=gpd.points_from_xy(stops_low["stop_lon"], stops_low["stop_lat"]),
                                 crs="EPSG:4326")

# 3) Reproject to Web Mercator for basemap tiles
g_3857 = g.to_crs(epsg=3857)
stops_low_3857 = stops_low_gdf.to_crs(epsg=3857)

# 4) Static map A: continuous BusFreqScore

# Continuous choropleth WITHOUT scheme=  (so no .lower() anywhere)
fig, ax = plt.subplots(figsize=(10, 10), dpi=200)

vals = g_3857["BusFreqScore"].astype(float).replace([np.inf, -np.inf], np.nan)
vmin = float(vals.min()) if np.isfinite(vals.min()) else 0.0
vmax = float(vals.max()) if np.isfinite(vals.max()) else 1.0
if vmin == vmax:
    vmax = vmin + 1e-6  # avoid 0 range

g_3857.plot(
    column="BusFreqScore",
    cmap="YlOrRd",
    vmin=vmin, vmax=vmax,     # <- continuous scale
    linewidth=0.2,
    edgecolor="white",
    legend=True,              # matplotlib colorbar
    legend_kwds={"label": "Median departures/hour", "shrink": 0.6},
    ax=ax,
)

# overlay low-frequency stops
stops_low_3857.plot(ax=ax, markersize=3, color="#222222", alpha=0.6)

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.set_axis_off()
ax.set_title("Lancashire — Bus Frequency (Median departures/hour) with <1 dep/hr stops", pad=12)
plt.tight_layout()
fig.savefig(OUT_PNG_CONT, bbox_inches="tight")
plt.close(fig)



# 5) Static map B: categorical BusFreqBand
fig, ax = plt.subplots(figsize=(10, 10), dpi=200)
band_order = ["Extreme Desert","Desert","Low","Moderate","Good","High","Unknown"]
g_3857["BusFreqBand"] = pd.Categorical(g_3857["BusFreqBand"], categories=band_order, ordered=True)
# discrete colors
band_colors = {
    "Extreme Desert":"#4d004b",
    "Desert":"#8c6bb1",
    "Low":"#9ebcda",
    "Moderate":"#bfd3e6",
    "Good":"#e0ecf4",
    "High":"#f7fcfd",
    "Unknown":"#cccccc"
}
g_3857.plot(column="BusFreqBand",
            categorical=True,
            legend=True,
            cmap=None,
            color=g_3857["BusFreqBand"].map(band_colors),
            linewidth=0.2,
            edgecolor="white",
            ax=ax)
stops_low_3857.plot(ax=ax, markersize=3, color="#222222", alpha=0.6)

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.set_axis_off()
ax.set_title("Lancashire — Bus Frequency Bands (with <1 dep/hr stops)", pad=12)
plt.tight_layout()
fig.savefig(OUT_PNG_CAT, bbox_inches="tight")
plt.close(fig)

print(f"Saved PNGs:\n  {OUT_PNG_CONT}\n  {OUT_PNG_CAT}")

# 6) Interactive Folium map
#   Use WGS84 (EPSG:4326) for web mapping
g_4326 = g.to_crs(epsg=4326)
stops_low_4326 = stops_low_gdf.to_crs(epsg=4326)


# --- Robust breaks for Folium ---
vals = g_4326["BusFreqScore"].astype(float).replace([np.inf, -np.inf], np.nan).dropna()

if vals.empty:
    # nothing to map; make a dummy 4-break scale (3 classes)
    vmin, vmax = 0.0, 1.0
    mid1 = vmin + (vmax - vmin) / 3
    mid2 = vmin + 2*(vmax - vmin) / 3
    breaks = [vmin, mid1, mid2, vmax]
else:
    vmin, vmax = float(vals.min()), float(vals.max())
    if vmin == vmax:
        # all equal -> make a tiny span around vmin with 4 breaks
        eps = max(1e-6, abs(vmax)*1e-6 or 1e-6)
        breaks = [vmin - eps, vmin, vmin + eps, vmin + 2*eps]
    else:
        # try quantiles, but adapt k to unique values and ensure strict monotonicity
        uniq = int(vals.nunique())
        k = min(5, max(3, uniq))  # at least 3 classes
        q = mc.Quantiles(vals, k=k)
        bins = list(q.bins)

        # build full threshold scale = [min, ...bins..., max+epsilon]
        eps = max(1e-9, abs(vmax)*1e-9)
        breaks = [vmin] + bins
        breaks = np.unique(breaks).tolist()

        # ensure top edge > vmax
        if breaks[-1] <= vmax:
            breaks[-1] = vmax + eps

        # 🔴 IMPORTANT: ensure at least 4 breakpoints (3 classes)
        while len(breaks) < 4:
            mid = (breaks[0] + breaks[-1]) / 2
            breaks = sorted(set(breaks + [mid]))

# (optional debug)
print("BusFreqScore min/max:", vmin, vmax)
print("Breaks used for Folium:", breaks, "len:", len(breaks))

# --- Folium map ---
m = folium.Map(location=[53.77, -2.7], zoom_start=9, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=g_4326.to_json(),
    data=g_4326[["LSOA11CD","BusFreqScore"]],
    columns=["LSOA11CD", "BusFreqScore"],
    key_on="feature.properties.LSOA11CD",
    fill_color="YlOrRd",
    bins=breaks,   # now guaranteed len(breaks) >= 4
    fill_opacity=0.8,
    line_opacity=0.2,
    legend_name="Median departures/hour",
    highlight=True,
).add_to(m)

# Optional: hover tooltip layer
tooltip_fields = ["LSOA11NM","LADNM","stops_count","median_departures_per_hour",
                  "avg_departures_per_hour","pct_stops_lt1_per_hour","BusFreqBand"]
tooltip_alias  = ["LSOA","District","Stops","Median dep/hr","Avg dep/hr","% <1/hr","Band"]

folium.GeoJson(
    g_4326.to_json(),
    name="LSOAs",
    style_function=lambda x: {"fillOpacity": 0, "color": "transparent"},
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_alias, sticky=False),
).add_to(m)

# overlay <1 dep/hr stops (if you computed stops_low_4326)
for _, r in stops_low_4326.iterrows():
    folium.CircleMarker(
        location=(r.geometry.y, r.geometry.x),
        radius=2.5,
        color="#111111",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Stop {r['stop_id']} — {r['departures_per_hour']:.2f} dep/hr",
    ).add_to(m)

m.save(OUT_HTML)
print(f"Saved interactive map: {OUT_HTML}\nBreaks used: {breaks}")







# 7) Power BI export (flat CSV)
pbi_cols = [
    "LSOA11CD","LSOA11NM","LADNM",
    "stops_count","median_departures_per_hour","avg_departures_per_hour",
    "pct_stops_lt1_per_hour","BusFreqBand","BusFreq_norm"
]
g[pbi_cols].to_csv(OUT_PBI_CSV, index=False)
print(f"Saved Power BI CSV: {OUT_PBI_CSV}")


/var/folders/d8/hkx8jz3s4_zbp6_5ff82k80h0000gn/T/ipykernel_19204/328536713.py:83: UserWarning: Only specify one of 'column' or 'color'. Using 'color'.
  g_3857.plot(column="BusFreqBand",


Saved PNGs:
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/map_busfreq_continuous.png
  /Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/map_busfreq_band.png


/Users/shrankhlagarg/miniforge3/envs/geo_env/lib/python3.10/site-packages/mapclassify/classifiers.py:1653: UserWarning: Not enough unique values in array to form 5 classes. Setting k to 2.
  self.bins = quantile(y, k=k)
